# Multimodal input preparation


In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
while not (PROJECT_ROOT / "modeling_pipeline.py").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Run this code from the MMDLPC_Code_PDF folder.")
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT.parent / "MMDLPC_Code_PDF_Data"
OUTPUT_DIR = PROJECT_ROOT / '04_Multimodal/00_Preprocessing'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXTERNAL_INPUT_DIR = Path(os.environ.get('MMDLPC_EXTERNAL_INPUTS', str(DATA_ROOT / 'External_Inputs')))


## Clinical Pathology


In [ ]:
import pandas as pd
import numpy as np
import os
from collections import namedtuple
from onekey_algo import OnekeyDS as okds
from onekey_algo import get_param_in_cwd
import onekey_algo.custom.components as okcomp
from onekey_algo.custom.components.comp1 import fillna
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 300

# data_ = pd.read_csv(get_param_in_cwd('clinic_file') or okds.survival)
# data_['ID'] = data_['ID'].map(lambda x: f"{x}.nii.gz" if not (f"{x}".endswith('.nii.gz') or  f"{x}".endswith('.nii')) else x)
# group_info = pd.read_csv('group.csv')
# data_ = pd.merge(data_, group_info, on='ID', how='inner')
# sel_feature = get_param_in_cwd('clinic_sel_columns') or ['age', 'BMI', 'chemotherapy', 'gender', 'drink', 'smoke']
# sel_features = ['ID'] + sel_feature + ['group', 'label']
# data_ = fillna(data_)
# data_[sel_features].to_csv('clinic_sel.csv', index=False, encoding='utf_8_sig')

In [ ]:
# 读取数据，B超诊断阳性=1，bc_data.csv是要读取的数据。
data_file = str(DATA_ROOT / '01_Clinical_and_Cohort/02_Clinical_Model/clinical_sel.csv')
labels = ['HRR_ANY']
featrues_not_use = ['ID']

clinical_data = pd.read_csv(data_file, header=0)
clinical_data['ID'] = clinical_data['ID'].map(lambda x: str(x))
path_data = pd.read_csv(str(DATA_ROOT / '03_Pathomics/path_sel_features_reference.csv'))
path_data['ID'] = path_data['ID'].map(lambda x: str(x))

label_data = pd.read_csv(str(DATA_ROOT / '03_Pathomics/path_group_reference.csv'))
label_data['ID'] = label_data['ID'].map(lambda x: str(x))

# [['ID', 'group', 'Primary_Gleason', 'age_at_diagnosis'] + labels]
structed_data = pd.merge(pd.merge(path_data, clinical_data, on='ID', how='inner'), label_data['ID'], on='ID', how='inner')
# structed_data = pd.merge(path_data, clinical_data, on='ID', how='inner')
structed_data

In [ ]:
# 删掉ID这一列。
ids = structed_data['ID']
structed_data = structed_data.drop(featrues_not_use, axis=1)
structed_data

In [ ]:
structed_data.describe()

In [ ]:
# from onekey_algo.custom.components.comp1 import normalize_df
# data = normalize_df(structed_data, not_norm=labels + ['group'])
# data = data.dropna(axis=1)
# data.describe()
data = structed_data
data

In [ ]:
# 如果需要选择相关系数使用对应的相关系数即可。
# pearson_corr = data.corr('pearson')
# kendall_corr = data.corr('kendall')
spearman_corr = data.corr('spearman')

import seaborn as sns
import matplotlib.pyplot as plt
from onekey_algo.custom.components.comp1 import draw_matrix
plt.figure(figsize=(40.0, 30.0))

# 选择可视化的相关系数
draw_matrix(spearman_corr, annot=True, cmap='YlGnBu', cbar=False)
plt.savefig(str(OUTPUT_DIR / f'Clinical_Pathology_Clinic-Path_feature_corr.pdf'), bbox_inches = 'tight')

In [ ]:
sel_data = data

In [ ]:
import numpy as np
import onekey_algo.custom.components as okcomp

group_info = 'group'
n_classes = 2
train_data = sel_data[(sel_data[group_info] != 'CPGEA')]
train_ids = ids[train_data.index]
train_data = train_data.reset_index()
train_data = train_data.drop('index', axis=1)
y_data = train_data[labels]
X_data = train_data.drop(labels + [group_info], axis=1)

test_data = sel_data[sel_data[group_info] == 'CPGEA']
test_ids = ids[test_data.index]
test_data = test_data.reset_index()
test_data = test_data.drop('index', axis=1)
y_test_data = test_data[labels]
X_test_data = test_data.drop(labels + [group_info], axis=1)

y_all_data = sel_data[labels]
X_all_data = sel_data.drop(labels + [group_info], axis=1)

column_names = X_data.columns
print(f"训练集样本数：{X_data.shape}, 测试集样本数：{X_test_data.shape}")

In [ ]:
y_test_data.value_counts()

## Clinical Radiology


In [ ]:
import pandas as pd
import numpy as np
import os
from collections import namedtuple
from onekey_algo import OnekeyDS as okds
from onekey_algo import get_param_in_cwd
import onekey_algo.custom.components as okcomp
from onekey_algo.custom.components.comp1 import fillna
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 300

# data_ = pd.read_csv(get_param_in_cwd('clinic_file') or okds.survival)
# data_['ID'] = data_['ID'].map(lambda x: f"{x}.nii.gz" if not (f"{x}".endswith('.nii.gz') or  f"{x}".endswith('.nii')) else x)
# group_info = pd.read_csv('group.csv')
# data_ = pd.merge(data_, group_info, on='ID', how='inner')
# sel_feature = get_param_in_cwd('clinic_sel_columns') or ['age', 'BMI', 'chemotherapy', 'gender', 'drink', 'smoke']
# sel_features = ['ID'] + sel_feature + ['group', 'label']
# data_ = fillna(data_)
# data_[sel_features].to_csv('clinic_sel.csv', index=False, encoding='utf_8_sig')

In [ ]:
# 读取数据，B超诊断阳性=1，bc_data.csv是要读取的数据。
data_file = str(DATA_ROOT / '01_Clinical_and_Cohort/02_Clinical_Model/clinical_sel.csv')
labels = ['HRR_ANY']
featrues_not_use = ['ID']

clinical_data = pd.read_csv(data_file, header=0)
clinical_data = clinical_data.drop('group', axis=1)
clinical_data['ID'] = clinical_data['ID'].map(lambda x: str(x) + '.nii.gz')
path_data = pd.read_csv(str(DATA_ROOT / '02_Radiomics/02_Modeling_and_Evaluation/rad_sel_features_reference.csv'))
path_data['ID'] = path_data['ID'].map(lambda x: str(x))

label_data = pd.read_csv(str(DATA_ROOT / '02_Radiomics/02_Modeling_and_Evaluation/radiomics_group_reference.csv'))
label_data['ID'] = label_data['ID'].map(lambda x: str(x))

# [['ID', 'group', 'Primary_Gleason', 'age_at_diagnosis'] + labels]
structed_data = pd.merge(pd.merge(path_data, clinical_data, on='ID', how='inner'), label_data[['ID', 'group']], on='ID', how='inner')
# structed_data = pd.merge(path_data, clinical_data, on='ID', how='inner')
structed_data

In [ ]:
# 删掉ID这一列。
ids = structed_data['ID']
structed_data = structed_data.drop(featrues_not_use, axis=1)
structed_data

In [ ]:
structed_data.describe()

In [ ]:
# from onekey_algo.custom.components.comp1 import normalize_df
# data = normalize_df(structed_data, not_norm=labels + ['group'])
# data = data.dropna(axis=1)
# data.describe()
data = structed_data
data

In [ ]:
# 如果需要选择相关系数使用对应的相关系数即可。
# pearson_corr = data.corr('pearson')
# kendall_corr = data.corr('kendall')
spearman_corr = data.corr('spearman')

import seaborn as sns
import matplotlib.pyplot as plt
from onekey_algo.custom.components.comp1 import draw_matrix
plt.figure(figsize=(20.0, 16.0))

# 选择可视化的相关系数
draw_matrix(spearman_corr, annot=True, cmap='YlGnBu', cbar=False)
plt.savefig(str(OUTPUT_DIR / f'Clinical_Radiology_Clinic-Rad_feature_corr.pdf'), bbox_inches = 'tight')

In [ ]:
sel_data = data
sel_data

In [ ]:
import numpy as np
import onekey_algo.custom.components as okcomp
from collections import OrderedDict

group_info = 'group'
n_classes = 2
train_data = sel_data[(sel_data[group_info] == 'train')]
train_ids = ids[train_data.index]
train_data = train_data.reset_index()
train_data = train_data.drop('index', axis=1)
y_data = train_data[labels]
X_data = train_data.drop(labels + [group_info], axis=1)

subsets = ['test']
val_datasets = OrderedDict()
for subset in subsets:
    val_data = sel_data[sel_data[group_info] == subset]
    val_ids = ids[val_data.index]
    val_data = val_data.reset_index()
    val_data = val_data.drop('index', axis=1)
    y_val_data = val_data[labels]
    X_val_data = val_data.drop(labels + [group_info], axis=1)
    val_datasets[subset] = [X_val_data, y_val_data, val_ids]

y_all_data = sel_data[labels]
X_all_data = sel_data.drop(labels + [group_info], axis=1)

column_names = X_data.columns
print(f"训练集样本数：{X_data.shape}，", '，'.join([f"{subset}样本数：{d_[0].shape}" for subset, d_ in val_datasets.items()]))

## Pathology Radiology


In [ ]:
import pandas as pd
import numpy as np
import os
from collections import namedtuple
from onekey_algo import OnekeyDS as okds
from onekey_algo import get_param_in_cwd
import onekey_algo.custom.components as okcomp
from onekey_algo.custom.components.comp1 import fillna
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 300

# data_ = pd.read_csv(get_param_in_cwd('clinic_file') or okds.survival)
# data_['ID'] = data_['ID'].map(lambda x: f"{x}.nii.gz" if not (f"{x}".endswith('.nii.gz') or  f"{x}".endswith('.nii')) else x)
# group_info = pd.read_csv('group.csv')
# data_ = pd.merge(data_, group_info, on='ID', how='inner')
# sel_feature = get_param_in_cwd('clinic_sel_columns') or ['age', 'BMI', 'chemotherapy', 'gender', 'drink', 'smoke']
# sel_features = ['ID'] + sel_feature + ['group', 'label']
# data_ = fillna(data_)
# data_[sel_features].to_csv('clinic_sel.csv', index=False, encoding='utf_8_sig')

In [ ]:
from onekey_algo.custom.utils import print_join_info

data_file = str(DATA_ROOT / '01_Clinical_and_Cohort/02_Clinical_Model/clinical_sel.csv')
labels = ['HRR_ANY']
featrues_not_use = ['ID']

clinical_data = pd.read_csv(data_file, header=0)[['ID', 'group']]
clinical_data = clinical_data[[c for c in clinical_data.columns if c != 'group' and c not in labels]]
clinical_data['ID'] = clinical_data['ID'].map(lambda x: str(x))
path_data = pd.read_csv(str(DATA_ROOT / '03_Pathomics/path_sel_features_reference.csv'))
path_data['ID'] = path_data['ID'].map(lambda x: str(x))
rad_data = pd.read_csv(str(DATA_ROOT / '02_Radiomics/02_Modeling_and_Evaluation/rad_sel_features_reference.csv'))
rad_data['ID'] = rad_data['ID'].map(lambda x: x.replace('.nii.gz', ''))

label_data = pd.read_csv(str(DATA_ROOT / '02_Radiomics/02_Modeling_and_Evaluation/radiomics_group_reference.csv'))
label_data['ID'] = label_data['ID'].map(lambda x: x.replace('.nii.gz', ''))
path_label_data = pd.read_csv(str(DATA_ROOT / '03_Pathomics/path_group_reference.csv'))
path_label_data['ID'] = path_label_data['ID'].map(lambda x: str(x))
all_label_data = pd.merge(label_data, path_label_data['ID'], on='ID', how='inner')

features_data = pd.merge(pd.merge(rad_data, path_data, on='ID', how='inner'), clinical_data, on='ID', how='inner')
structed_data = pd.merge(features_data, all_label_data, on='ID', how='inner')
print_join_info(path_label_data, label_data)
# structed_data = pd.merge(path_data, clinical_data, on='ID', how='inner')
structed_data

In [ ]:
structed_data.columns

In [ ]:
# 删掉ID这一列。
ids = structed_data['ID']
structed_data = structed_data.drop(featrues_not_use, axis=1)
structed_data

In [ ]:
structed_data.describe()

In [ ]:
# from onekey_algo.custom.components.comp1 import normalize_df
# data = normalize_df(structed_data, not_norm=labels + ['group'])
# data = data.dropna(axis=1)
# data.describe()
data = structed_data
data

In [ ]:
# 如果需要选择相关系数使用对应的相关系数即可。
# pearson_corr = data.corr('pearson')
# kendall_corr = data.corr('kendall')
spearman_corr = data[[c for c in data.columns if c !='IM']].corr('spearman')

import seaborn as sns
import matplotlib.pyplot as plt
from onekey_algo.custom.components.comp1 import draw_matrix
plt.figure(figsize=(50.0, 40.0))

# 选择可视化的相关系数
draw_matrix(spearman_corr, annot=True, cmap='YlGnBu', cbar=False)
plt.savefig(str(OUTPUT_DIR / f'Pathology_Radiology_Path-Rad_feature_corr.pdf'), bbox_inches = 'tight')

In [ ]:
sel_data = data
import numpy as np
import onekey_algo.custom.components as okcomp

n_classes = 2
sel_data = sel_data.drop(['group'], axis=1)
y_data = sel_data[labels]
X_data = sel_data.drop(labels, axis=1)
column_names = X_data.columns

X_train, X_test, y_train, y_test = okcomp.comp1.split_dataset(X_data, y_data, test_size=0.3)
print(f"训练集样本数：{X_train.shape}, 验证集样本数：{X_test.shape}")

## Clinical Pathology Radiology


In [ ]:
import pandas as pd
import numpy as np
import os
from collections import namedtuple
from onekey_algo import OnekeyDS as okds
from onekey_algo import get_param_in_cwd
import onekey_algo.custom.components as okcomp
from onekey_algo.custom.components.comp1 import fillna
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 300

# data_ = pd.read_csv(get_param_in_cwd('clinic_file') or okds.survival)
# data_['ID'] = data_['ID'].map(lambda x: f"{x}.nii.gz" if not (f"{x}".endswith('.nii.gz') or  f"{x}".endswith('.nii')) else x)
# group_info = pd.read_csv('group.csv')
# data_ = pd.merge(data_, group_info, on='ID', how='inner')
# sel_feature = get_param_in_cwd('clinic_sel_columns') or ['age', 'BMI', 'chemotherapy', 'gender', 'drink', 'smoke']
# sel_features = ['ID'] + sel_feature + ['group', 'label']
# data_ = fillna(data_)
# data_[sel_features].to_csv('clinic_sel.csv', index=False, encoding='utf_8_sig')

In [ ]:
from onekey_algo.custom.utils import print_join_info

data_file = str(DATA_ROOT / '01_Clinical_and_Cohort/02_Clinical_Model/clinical_sel.csv')
labels = ['HRR_ANY']
featrues_not_use = ['ID']

clinical_data = pd.read_csv(data_file, header=0)#[['ID', 'group']]
clinical_data = clinical_data[[c for c in clinical_data.columns if c != 'group' and c not in labels]]
clinical_data['ID'] = clinical_data['ID'].map(lambda x: str(x))
path_data = pd.read_csv(str(DATA_ROOT / '03_Pathomics/path_sel_features_reference.csv'))
path_data['ID'] = path_data['ID'].map(lambda x: str(x))
rad_data = pd.read_csv(str(DATA_ROOT / '02_Radiomics/02_Modeling_and_Evaluation/rad_sel_features_reference.csv'))
rad_data['ID'] = rad_data['ID'].map(lambda x: x.replace('.nii.gz', ''))

label_data = pd.read_csv(str(DATA_ROOT / '02_Radiomics/02_Modeling_and_Evaluation/radiomics_group_reference.csv'))
label_data['ID'] = label_data['ID'].map(lambda x: x.replace('.nii.gz', ''))
path_label_data = pd.read_csv(str(DATA_ROOT / '03_Pathomics/path_group_reference.csv'))
path_label_data['ID'] = path_label_data['ID'].map(lambda x: str(x))
all_label_data = pd.merge(label_data, path_label_data['ID'], on='ID', how='inner')

features_data = pd.merge(pd.merge(rad_data, path_data, on='ID', how='inner'), clinical_data, on='ID', how='inner')
structed_data = pd.merge(features_data, all_label_data, on='ID', how='inner')
print_join_info(path_label_data, label_data)
# structed_data = pd.merge(path_data, clinical_data, on='ID', how='inner')
structed_data

In [ ]:
structed_data.columns

In [ ]:
# 删掉ID这一列。
ids = structed_data['ID']
structed_data = structed_data.drop(featrues_not_use, axis=1)
structed_data

In [ ]:
structed_data.describe()

In [ ]:
# from onekey_algo.custom.components.comp1 import normalize_df
# data = normalize_df(structed_data, not_norm=labels + ['group'])
# data = data.dropna(axis=1)
# data.describe()
data = structed_data
data

In [ ]:
# 如果需要选择相关系数使用对应的相关系数即可。
# pearson_corr = data.corr('pearson')
# kendall_corr = data.corr('kendall')
spearman_corr = data[[c for c in data.columns if c !='IM']].corr('spearman')

import seaborn as sns
import matplotlib.pyplot as plt
from onekey_algo.custom.components.comp1 import draw_matrix
plt.figure(figsize=(50.0, 40.0))

# 选择可视化的相关系数
draw_matrix(spearman_corr, annot=True, cmap='YlGnBu', cbar=False)
plt.savefig(str(OUTPUT_DIR / f'Clinical_Pathology_Radiology_Combined_feature_corr.pdf'), bbox_inches = 'tight')

In [ ]:
sel_data = data
import numpy as np
import onekey_algo.custom.components as okcomp

n_classes = 2
sel_data = sel_data.drop(['group'], axis=1)
y_data = sel_data[labels]
X_data = sel_data.drop(labels, axis=1)
column_names = X_data.columns

X_train, X_test, y_train, y_test = okcomp.comp1.split_dataset(X_data, y_data, test_size=0.3)
print(f"训练集样本数：{X_train.shape}, 验证集样本数：{X_test.shape}")